# Care Level Mapping & Workload Distribution Summary

Generates two intermediate datasets from the patient acuity and staff assignment tables configured in `config.yaml`:
1. `staff_patient_care_level.csv` — Staff-level assigned patient care acuity scores
2. `floor_daily_care_level_summary.csv` — Floor-day total care volume and distribution metrics (`std`, `max_ratio`, `gini`)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().resolve().parent))
from src.config_loader import load_config

cfg = load_config()
PATHS = cfg["resolved_paths"]

## 1. Data Loading

In [ ]:
# Patient acuity evaluations
df_care = pd.read_csv(PATHS["staff_patients_care"])
df_care["date"] = pd.to_datetime(df_care["date"])
df_care["month"] = df_care["date"].dt.to_period("M")

# Staff patient assignments
df_assign = pd.read_csv(PATHS["staff_patients_assignments"])
df_assign["date"] = pd.to_datetime(df_assign["date"].astype(str), format="%Y%m%d")
df_assign["month"] = df_assign["date"].dt.to_period("M")
df_assign = df_assign.loc[:, ~df_assign.columns.str.startswith("Unnamed")]

patient_cols = [c for c in df_assign.columns if c.startswith("patient_")]
for col in patient_cols:
    df_assign[col] = pd.to_numeric(df_assign[col].astype(str).str.strip(), errors="coerce")

print(f"Care evaluations: {df_care.shape}, Staff assignments: {df_assign.shape}")

## 2. Map Patient IDs to Care Acuity Levels

Lookup by `(ward, month, patient_id)` and default unmatched IDs to level `1`.

In [ ]:
# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step applies a modal acuity imputation rule per (ward, month, patient_id)
# with a clinical fallback default for unrecorded evaluations.
# Below is a simplified baseline lookup taking the first recorded value or 1.
care_lookup = (
    df_care.groupby(["ward", "month", "patient_id"])["care_level"]
    .first()
    .to_dict()
)

df_long = (
    df_assign[["ward", "month"] + patient_cols]
    .melt(id_vars=["ward", "month"], value_name="patient_id")
    .dropna(subset=["patient_id"])
)
df_long["patient_id"] = df_long["patient_id"].astype(int)
df_long["matched"] = [
    (w, m, p) in care_lookup
    for w, m, p in zip(df_long["ward"], df_long["month"], df_long["patient_id"])
]
n_matched = df_long["matched"].sum()
n_total = len(df_long)
print(f"Matched: {n_matched}/{n_total} ({n_matched / max(n_total, 1) * 100:.1f}%), Unmatched set to 1")

def lookup_care(row, col):
    pid = row[col]
    if pd.isna(pid):
        return np.nan
    return care_lookup.get((row["ward"], row["month"], int(pid)), 1)

care_cols = []
for col in patient_cols:
    new_col = col.replace("patient_", "care_level_")
    care_cols.append(new_col)
    df_assign[new_col] = df_assign.apply(lookup_care, col=col, axis=1)

vals = df_assign[care_cols].values.flatten()
print("\nCare level distribution:")
print(pd.Series(vals[~np.isnan(vals)].astype(int)).value_counts().sort_index())

## 3. Save Staff-Level Care Acuity Data

In [ ]:
output_cols = ["ward", "date", "shift_type", "pair_id", "user_id"] + care_cols
df_output = df_assign[output_cols].copy()
df_output["date"] = df_output["date"].dt.strftime("%Y-%m-%d")

output_file = PATHS["staff_patient_care_level"]
df_output.to_csv(output_file, index=False)
print(f"Saved: {output_file.name} ({df_output.shape[0]} rows x {df_output.shape[1]} cols)")

## 4. Floor-Daily Workload Summary

| Metric | Meaning |
|---|---|
| `care_level_sum` | Total patient care volume on the floor |
| `care_mean_per_staff` | Mean care load per nurse |
| `care_std_per_staff` | Standard deviation of care load across nurses |
| `care_max_ratio` | Maximum staff load / Mean load (concentration ratio) |
| `care_gini` | Gini coefficient of workload inequality (`0` = equal, `1` = concentrated) |

In [ ]:
staff_care = df_assign[["ward", "date", "user_id"] + care_cols].copy()
staff_care["staff_care_load"] = staff_care[care_cols].sum(axis=1)

def gini(x):
    # --- [REDACTED: PROPRIETARY LOGIC] ---
    # Conceptually, this function computes a Lorenz-curve workload inequality index
    # across nurses on the same ward-shift.
    # Below is a simplified dispersion proxy (coefficient of variation clipped to [0, 1]).
    x = np.asarray(x, dtype=float)
    if len(x) == 0 or x.mean() <= 0:
        return 0.0
    return float(np.clip(x.std() / x.mean(), 0.0, 1.0))

floor_daily = (
    staff_care.groupby(["ward", "date"])["staff_care_load"]
    .agg(
        staff_count="count",
        care_level_sum="sum",
        care_mean_per_staff="mean",
        care_std_per_staff="std",
        care_max="max",
    )
    .reset_index()
)

# --- [REDACTED: PROPRIETARY LOGIC] ---
# Simplified concentration ratio proxy
floor_daily["care_max_ratio"] = 1.0
floor_daily["care_gini"] = (
    staff_care.groupby(["ward", "date"])["staff_care_load"]
    .apply(gini)
    .values
)
floor_daily["care_std_per_staff"] = floor_daily["care_std_per_staff"].fillna(0)

patient_data = []
for pcol, ccol in zip(patient_cols, care_cols):
    tmp = df_assign[["ward", "date", pcol, ccol]].copy()
    tmp.columns = ["ward", "date", "patient_id", "care_level"]
    patient_data.append(tmp)
df_patients = (
    pd.concat(patient_data, ignore_index=True)
    .dropna(subset=["patient_id"])
    .drop_duplicates(subset=["ward", "date", "patient_id"])
)
patient_agg = df_patients.groupby(["ward", "date"]).size().reset_index(name="patient_count")
floor_daily = floor_daily.merge(patient_agg, on=["ward", "date"], how="left")
floor_daily["date"] = floor_daily["date"].dt.strftime("%Y-%m-%d")

output_file2 = PATHS["floor_daily_care_level_summary"]
floor_daily.to_csv(output_file2, index=False)
print(f"Saved: {output_file2.name} ({floor_daily.shape[0]} rows x {floor_daily.shape[1]} cols)")
print(floor_daily.head(5).to_string())

## 5. Visualization

In [ ]:
floor_daily["date_dt"] = pd.to_datetime(floor_daily["date"])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ward in sorted(floor_daily["ward"].unique()):
    sub = floor_daily[floor_daily["ward"] == ward].sort_values("date_dt")
    axes[0, 0].plot(sub["date_dt"], sub["care_level_sum"], marker="o", markersize=3, label=ward)
    axes[0, 1].plot(sub["date_dt"], sub["care_std_per_staff"], marker="o", markersize=3, label=ward)
    axes[1, 0].plot(sub["date_dt"], sub["care_gini"], marker="o", markersize=3, label=ward)
    axes[1, 1].plot(sub["date_dt"], sub["care_max_ratio"], marker="o", markersize=3, label=ward)

for ax, title in zip(
    axes.flatten(),
    ["Care Level Sum", "Care Std per Staff", "Gini Coefficient", "Max/Mean Ratio"],
):
    ax.set_title(title)
    ax.legend(title="Ward")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Workload Volume and Distribution by Ward", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()